# Agilent LCR & Janis Temperature Controller
Automated Measurements with Live Plotting

In [1]:
# Imports
from nfoinstruments.drivers import Janis, E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures import (
    set_temperature_and_wait, 
    sweep_frequency_lcr,
    single_frequency_time_scan,
    set_bias_and_wait,
    build_cv_bias_path,
    sweep_cv_lcr,
    load_measurement_files,
    load_cv_measurement_files,
    plot_all_measurements,
    plot_measurement_comparison,
    plot_time_scan_comparison,
    plot_cv_comparison
)
from nfoinstruments.procedures.utils import run_temperature_bias_sweep_with_live_plot, run_cv_sweep_with_live_plot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Output configuration
output_path = Path(r"C:\Users\F110216\Desktop\Data_Horatio\UCL111")

In [2]:
# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB0::16::INSTR': Janis,
    'GPIB1::17::INSTR': E4890A
})

janis = mm.devices['GPIB0::16::INSTR']
lcr = mm.devices['GPIB1::17::INSTR']

# Configure LCR meter
lcr.measurement_type = E4890A.MeasurementType.ZTD  # Options: ZTD (Z, θ), RX (R, X), CPD (Cp, D), CSD (Cs, D)...
lcr.measurement_time = E4890A.MeasurementTime.LONG # Options: SHORT, MEDIUM, LONG
lcr.averages = 1                                 # 1 to 256
lcr.signal_type = E4890A.SignalType.VOLTAGE        # VOLTAGE or CURRENT
lcr.alc_enabled = True                             # Automatic Level Control (True/False)

# Initialize into Standby Mode (Protect Sample during probe landing)
print("Initializing LCR into STANDBY MODE...")
lcr.signal_amplitude = 0.0                       # Signal Amplitude (V or A)
lcr.bias = 0.0                                     # DC Bias (V)
lcr.trigger_source = 'HOLD'                        # Stop internal measurements (protects from bridge overload)

print(f"LCR configured:")
print(f"  Type: {lcr.measurement_type.name}")
print(f"  Time: {lcr.measurement_time.name} (Avg: {lcr.averages})")
print(f"  ALC: {lcr.alc_enabled}")
print(f"  AC Signal: {lcr.signal_amplitude} V")
print(f"  DC Bias: {lcr.bias} V")
print(f"  Trigger: {lcr.trigger_source}")

['GPIB0::15::INSTR', 'GPIB0::17::INSTR', 'GPIB0::18::INSTR', 'GPIB0::22::INSTR', 'GPIB1::15::INSTR', 'GPIB1::17::INSTR', 'GPIB0::16::INSTR']
LCR Connected: Agilent Technologies,E4980A,MY46204018,A.06.17
{'_alc_enabled': True,
 '_averages': 1,
 '_bias': 0,
 '_frequency': 100,
 '_measurement_time': <MeasurementTime.MEDIUM: 'MED'>,
 '_measurement_type': <MeasurementType.RX: 'RX'>,
 '_signal_amplitude': 1,
 '_signal_type': <SignalType.VOLTAGE: 2>,
 '_trigger_source': 'INT',
 'address': 'GPIB1::17::INSTR',
 'measurement_timeout': 20,
 'resource': <'GPIBInstrument'('GPIB1::17::INSTR')>}
Initializing LCR into STANDBY MODE...
LCR configured:
  Type: ZTD
  Time: LONG (Avg: 1)
  ALC: True
  AC Signal: 0.0 V
  DC Bias: 0.0 V
  Trigger: HOLD


## Manual LCR Control
Run this cell to manually activate or deactivate the LCR output if needed for setup.

In [3]:
# --- MANUAL STANDBY TOGGLE ---

# Uncomment to ACTIVATE signal
# lcr.signal_amplitude = 0.05
# lcr.trigger_source = 'INT'
# print(f"Signal ACTIVE: {lcr.signal_amplitude} V, Trigger: INT")

# Uncomment to DEACTIVATE signal (Standby)
# lcr.signal_amplitude = 0.0
# lcr.bias = 0.0
# lcr.trigger_source = 'HOLD'
# print(f"Signal DEACTIVATED (Standby mode: 0V, HOLD)")


## 1. Initial (Pristine) IS Sweep
Do a single sweep at the current temperature before the main loop.

In [ ]:
# Define parameters for Pristine Sweep
# 60 points per decade, from 20 Hz to 2 MHz (5 decades -> 300 points)
frequencies = np.round(np.logspace(np.log10(20), np.log10(2e6), num=300), decimals=2)

# Use the current temperature of the Janis controller
current_temp = janis.temperature
dc_bias = 0.0  # V
ac_amplitude = 0.05 # V

sweep_name = "dev8_I_pristine"

# Run pristine measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=[current_temp],  # Single temperature point
    bias_points=[dc_bias],       # Single bias point
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    freq_points=frequencies,
    Vrms=ac_amplitude,           # Function will safely turn AC on, then off again at the end
    filename_suffix="_pristine", # Appended to filename
    run_count_start=1
)

Activating LCR signal amplitude to 0.05 V and trigger INT...

Temperature: 299.998 K
Setting temperature to 299.998 K...
Waiting for temperature stability...

Temperature stable at 300.00 K
Additional 30s settling time...

  Bias: +0.00 V
  Measuring -> run001_temp_300_DC_pos0.00V_pristine.csv ...


## 2. Main Temperature & Bias Loop (IS Scans)

In [ ]:
# Define parameters for IS Sweep
# Start at 300K, stop at 150K (so we set limit to 140), step by -10K
temperatures = list(range(300, 140, -10))
dc_biases = [-2.0, -1.0, 0.0, 1.0, 2.0]  # V
ac_amplitude = 0.05 # V

# Run measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=temperatures,
    bias_points=dc_biases,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    freq_points=frequencies,
    Vrms=ac_amplitude,
    filename_suffix="", 
    run_count_start=next_run
)

## 3. Capacitance-Voltage (CV) Scans

In [ ]:
# Define parameters for CV Sweep
# CV loop shape: 0 -> +Vmax -> Vmin -> 0
Vmin = -5.0
Vmax = 5.0
Vstep = 0.5

# AC condition 
Vrms = 0.1

# Frequency list for CV scans
cv_freq_points = [1e4, 1e5]

# Set up cycling to repeat the measurement with the same conditions 
cycles = 1 # Number of times to repeat the entire CV sequence at each temperature

# Run measurement
next_run = run_cv_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name="CV_" + sweep_name,
    temp_points=temperatures,
    freq_points=cv_freq_points,
    Vmin=Vmin,
    Vmax=Vmax,
    Vstep=Vstep,
    Vrms=Vrms,
    cycles=cycles,
    janis_ctrl=janis,
    lcr_ctrl=lcr,
    filename_suffix="",
    run_count_start=next_run
)